In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper to convert epoch-second min/max to ISO strings in UTC+8
def ts_range_iso(df, ts_col="timestamp"):
    if ts_col not in df or df[ts_col].dropna().shape[0] == 0:
        return None, None
    # convert series to timezone-aware datetimes then get min/max
    times = pd.to_datetime(df[ts_col], unit="s", utc=True).dt.tz_convert(tz)
    mn = times.min()
    mx = times.max()
    return mn.isoformat(), mx.isoformat()

# Helper to convert value_counts to list of dicts with python types, limited to n
def vc_to_list(vc, n=None, key_name="key", value_name="count"):
    if n is not None:
        vc = vc.head(n)
    df = vc.reset_index()
    df.columns = [key_name, value_name]
    # ensure native python types
    records = []
    for _, row in df.iterrows():
        records.append({key_name: None if pd.isna(row[key_name]) else str(row[key_name]),
                        value_name: int(row[value_name])})
    return records

# 1) General per-file summary
metric_rows = int(metric_df.shape[0])
metric_min_ts, metric_max_ts = ts_range_iso(metric_df, "timestamp")

trace_rows = int(trace_df.shape[0])
trace_min_ts, trace_max_ts = ts_range_iso(trace_df, "timestamp")

log_rows = int(log_df.shape[0])
log_min_ts, log_max_ts = ts_range_iso(log_df, "timestamp")

error_rows = int(error_logs_df.shape[0])
error_min_ts, error_max_ts = ts_range_iso(error_logs_df, "timestamp")

# 2) metric.csv specifics
metric_top_cmdb = vc_to_list(metric_df['cmdb_id'].value_counts(), n=20, key_name="cmdb_id", value_name="count")
# unique kpi_name values with counts, limit to top 50 if more
if 'kpi_name' in metric_df:
    metric_kpi_counts = vc_to_list(metric_df['kpi_name'].value_counts(), n=50, key_name="kpi_name", value_name="count")
else:
    metric_kpi_counts = []

# 3) trace.csv specifics
trace_top_cmdb = vc_to_list(trace_df['cmdb_id'].value_counts(), n=20, key_name="cmdb_id", value_name="count")
if 'trace_name' in trace_df:
    trace_name_counts = vc_to_list(trace_df['trace_name'].value_counts(), n=50, key_name="trace_name", value_name="count")
else:
    trace_name_counts = []

# 4) log.csv specifics
log_top_cmdb = vc_to_list(log_df['cmdb_id'].value_counts(), n=20, key_name="cmdb_id", value_name="count")
if 'log_name' in log_df:
    log_name_counts = vc_to_list(log_df['log_name'].value_counts(), n=50, key_name="log_name", value_name="count")
else:
    log_name_counts = []

# 5) error_logs.csv specifics
error_top_cmdb = vc_to_list(error_logs_df['cmdb_id'].value_counts(), n=20, key_name="cmdb_id", value_name="count")

# compute message prefix (first 100 chars) counts, top 10
if 'message' in error_logs_df:
    prefixes = error_logs_df['message'].fillna("").astype(str).str.slice(0, 100)
    error_msg_prefix_counts = vc_to_list(prefixes.value_counts(), n=10, key_name="message_prefix", value_name="count")
else:
    error_msg_prefix_counts = []

# Assemble JSON-serializable summary
summary = {
    "metric_summary": {
        "general": {"rows": metric_rows, "min_timestamp": metric_min_ts, "max_timestamp": metric_max_ts},
        "top_cmdb_ids": metric_top_cmdb,
        "kpi_name_counts_top50": metric_kpi_counts
    },
    "trace_summary": {
        "general": {"rows": trace_rows, "min_timestamp": trace_min_ts, "max_timestamp": trace_max_ts},
        "top_cmdb_ids": trace_top_cmdb,
        "trace_name_counts_top50": trace_name_counts
    },
    "log_summary": {
        "general": {"rows": log_rows, "min_timestamp": log_min_ts, "max_timestamp": log_max_ts},
        "top_cmdb_ids": log_top_cmdb,
        "log_name_counts_top50": log_name_counts
    },
    "error_logs_summary": {
        "general": {"rows": error_rows, "min_timestamp": error_min_ts, "max_timestamp": error_max_ts},
        "top_cmdb_ids": error_top_cmdb,
        "top_message_prefixes_100chars_top10": error_msg_prefix_counts
    }
}

summary

```
Out[1]:
```
summary = (
    "Summary of telemetry (UTC+8 time):\n"
    "- metric.csv: 145,452 rows from 2022-03-20T22:30:00+08:00 to 2022-03-20T22:59:00+08:00. "
    "Top cmdb_ids by volume: adservice (10230), adservice2 (9990), and many frontend and checkoutservice pods. "
    "Top KPI names are heavily populated for container.node-5 (network, CPU, memory, filesystem metrics).\n"
    "- trace.csv: 30,480 rows same time window. "
    "Top cmdb_ids: frontend-2 (2708), frontend-0 (2424), frontend-1 (2340), and checkoutservice pods. "
    "Most frequent trace metrics are frontend-related (duration, p95, error_rate, row_count) and traces to productcatalogservice.\n"
    "- log.csv: 1,768 rows from 22:30 to 22:59. "
    "Top cmdb_ids are many pods with ~60 rows each. Log names are dominated by log.error_count and log.row_count.\n"
    "- error_logs.csv: 689 rows from 2022-03-20T22:30:05+08:00 to 2022-03-20T22:59:57+08:00. "
    "Errors are concentrated in frontends (frontend-2: 463, frontend-0: 150, frontend-1: 76). "
    "All top error messages share the same 100-char prefix: 'severity: warning, message: failed to retrieve ads' (count=689).\n\n"
    "Key takeaway: error activity is concentrated in the frontend pods—especially frontend-2—with a consistent 'failed to retrieve ads' warning. "
    "Investigate frontend-2 and the upstream ad retrieval path (adservice/adservice2 and related network or node-5 metrics) as the next step."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_summary': {'general': {'rows': 145452, 'min_timestamp': '2022-03-20T22:30:00+08:00', 'max_timestamp': '2022-03-20T22:59:00+08:00'}, 'top_cmdb_ids': [{'cmdb_id': 'adservice', 'count': 10230}, {'cmdb_id': 'adservice2', 'count': 9990}, {'cmdb_id': 'frontend2-0', 'count': 5040}, {'cmdb_id': 'frontend-0', 'count': 4800}, {'cmdb_id': 'frontend-1', 'count': 4680}, {'cmdb_id': 'frontend-2', 'count': 4380}, {'cmdb_id': 'checkoutservice-0', 'count': 3780}, {'cmdb_id': 'checkoutservice-1', 'count': 3540}, {'cmdb_id': 'checkoutservice-2', 'count': 3540}, {'cmdb_id': 'productcatalogservice-0', 'count': 3480}, {'cmdb_id': 'productcatalogservice-2', 'count': 3480}, {'cmdb_id': 'productcatalogservice-1', 'count': 3360}, {'cmdb_id': 'checkoutservice2-0', 'count': 3300}, {'cmdb_id': 'adservice2-0', 'count': 3060}, {'cmdb_id': 'productcatalogservice2-0', 'count': 2880}, {'cmdb_id': 'cartservice-1', 'count': 2880}, {'cmdb_id': 'cartservice-0', 'count': 2760}, {'cmdb_id': 'cartservice-2', 'count': 2640}, {'cmdb_id': 'cartservice2-0', 'count': 2640}, {'cmdb_id': 'adservice-0', 'count': 2580}], 'kpi_name_counts_top50': [{'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_spec_cpu_quota', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_shares', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_swap_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_start_time_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.iowaiting', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_period', 'count': 990}, {'kpi_name': 'container.node-5.container_threads', 'count': 990}, {'kpi_name': 'container.node-5.container_threads_max', 'count': 990}, {'kpi_name': 'container.node-5.container_ulimits_soft.max_open_files', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_reservation_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.uninterruptible', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_rss', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_usage_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_swap', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_working_set_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_last_seen', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_cache', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_usage_MB./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.stopped', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_write_seconds./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.running', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_sector_writes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgfault', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgmajfault', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgfault', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 990}]}, 'trace_summary': {'general': {'rows': 30480, 'min_timestamp': '2022-03-20T22:30:00+08:00', 'max_timestamp': '2022-03-20T22:59:00+08:00'}, 'top_cmdb_ids': [{'cmdb_id': 'frontend-2', 'count': 2708}, {'cmdb_id': 'frontend-0', 'count': 2424}, {'cmdb_id': 'frontend-1', 'count': 2340}, {'cmdb_id': 'checkoutservice-0', 'count': 1948}, {'cmdb_id': 'checkoutservice-2', 'count': 1920}, {'cmdb_id': 'checkoutservice-1', 'count': 1844}, {'cmdb_id': 'productcatalogservice-1', 'count': 1128}, {'cmdb_id': 'productcatalogservice-0', 'count': 1108}, {'cmdb_id': 'productcatalogservice-2', 'count': 1108}, {'cmdb_id': 'frontend2-0', 'count': 996}, {'cmdb_id': 'recommendationservice-0', 'count': 840}, {'cmdb_id': 'recommendationservice-1', 'count': 840}, {'cmdb_id': 'recommendationservice-2', 'count': 840}, {'cmdb_id': 'cartservice-1', 'count': 772}, {'cmdb_id': 'cartservice-2', 'count': 764}, {'cmdb_id': 'cartservice-0', 'count': 744}, {'cmdb_id': 'currencyservice-0', 'count': 692}, {'cmdb_id': 'currencyservice-2', 'count': 688}, {'cmdb_id': 'currencyservice-1', 'count': 684}, {'cmdb_id': 'recommendationservice2-0', 'count': 600}], 'trace_name_counts_top50': [{'trace_name': 'trace.from_frontend-2.row_count', 'count': 617}, {'trace_name': 'trace.from_frontend-2.error_rate', 'count': 617}, {'trace_name': 'trace.from_frontend-2.duration_p95', 'count': 617}, {'trace_name': 'trace.from_frontend-2.duration_mean', 'count': 617}, {'trace_name': 'trace.from_frontend-0.duration_mean', 'count': 546}, {'trace_name': 'trace.from_frontend-0.duration_p95', 'count': 546}, {'trace_name': 'trace.from_frontend-0.error_rate', 'count': 546}, {'trace_name': 'trace.from_frontend-0.row_count', 'count': 546}, {'trace_name': 'trace.from_frontend-1.row_count', 'count': 525}, {'trace_name': 'trace.from_frontend-1.error_rate', 'count': 525}, {'trace_name': 'trace.from_frontend-1.duration_p95', 'count': 525}, {'trace_name': 'trace.from_frontend-1.duration_mean', 'count': 525}, {'trace_name': 'trace.self.row_count', 'count': 460}, {'trace_name': 'trace.self.error_rate', 'count': 460}, {'trace_name': 'trace.self.duration_p95', 'count': 460}, {'trace_name': 'trace.self.duration_mean', 'count': 460}, {'trace_name': 'trace.from_checkoutservice-0.row_count', 'count': 387}, {'trace_name': 'trace.from_checkoutservice-0.error_rate', 'count': 387}, {'trace_name': 'trace.from_checkoutservice-0.duration_p95', 'count': 387}, {'trace_name': 'trace.from_checkoutservice-0.duration_mean', 'count': 387}, {'trace_name': 'trace.from_checkoutservice-2.duration_mean', 'count': 382}, {'trace_name': 'trace.from_checkoutservice-2.duration_p95', 'count': 382}, {'trace_name': 'trace.from_checkoutservice-2.error_rate', 'count': 382}, {'trace_name': 'trace.from_checkoutservice-2.row_count', 'count': 382}, {'trace_name': 'trace.from_checkoutservice-1.duration_mean', 'count': 368}, {'trace_name': 'trace.from_checkoutservice-1.duration_p95', 'count': 368}, {'trace_name': 'trace.from_checkoutservice-1.error_rate', 'count': 368}, {'trace_name': 'trace.from_checkoutservice-1.row_count', 'count': 368}, {'trace_name': 'trace.to_productcatalogservice-1.row_count', 'count': 282}, {'trace_name': 'trace.to_productcatalogservice-1.error_rate', 'count': 282}, {'trace_name': 'trace.to_productcatalogservice-1.duration_p95', 'count': 282}, {'trace_name': 'trace.to_productcatalogservice-1.duration_mean', 'count': 282}, {'trace_name': 'trace.to_productcatalogservice-2.row_count', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-2.error_rate', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-2.duration_p95', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-2.duration_mean', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-0.row_count', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-0.error_rate', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-0.duration_p95', 'count': 277}, {'trace_name': 'trace.to_productcatalogservice-0.duration_mean', 'count': 277}, {'trace_name': 'trace.from_root.duration_mean', 'count': 220}, {'trace_name': 'trace.from_root.duration_p95', 'count': 220}, {'trace_name': 'trace.from_root.error_rate', 'count': 220}, {'trace_name': 'trace.from_root.row_count', 'count': 220}, {'trace_name': 'trace.from_frontend2-0.row_count', 'count': 189}, {'trace_name': 'trace.from_frontend2-0.error_rate', 'count': 189}, {'trace_name': 'trace.from_frontend2-0.duration_p95', 'count': 189}, {'trace_name': 'trace.from_frontend2-0.duration_mean', 'count': 189}, {'trace_name': 'trace.to_currencyservice-0.row_count', 'count': 173}, {'trace_name': 'trace.to_currencyservice-0.error_rate', 'count': 173}]}, 'log_summary': {'general': {'rows': 1768, 'min_timestamp': '2022-03-20T22:30:00+08:00', 'max_timestamp': '2022-03-20T22:59:00+08:00'}, 'top_cmdb_ids': [{'cmdb_id': 'adservice-0', 'count': 60}, {'cmdb_id': 'adservice-1', 'count': 60}, {'cmdb_id': 'adservice-2', 'count': 60}, {'cmdb_id': 'cartservice-0', 'count': 60}, {'cmdb_id': 'cartservice-1', 'count': 60}, {'cmdb_id': 'cartservice-2', 'count': 60}, {'cmdb_id': 'checkoutservice-0', 'count': 60}, {'cmdb_id': 'currencyservice-1', 'count': 60}, {'cmdb_id': 'currencyservice-0', 'count': 60}, {'cmdb_id': 'frontend-0', 'count': 60}, {'cmdb_id': 'currencyservice-2', 'count': 60}, {'cmdb_id': 'frontend-1', 'count': 60}, {'cmdb_id': 'productcatalogservice-2', 'count': 60}, {'cmdb_id': 'productcatalogservice-0', 'count': 60}, {'cmdb_id': 'frontend-2', 'count': 60}, {'cmdb_id': 'shippingservice-1', 'count': 60}, {'cmdb_id': 'shippingservice-2', 'count': 60}, {'cmdb_id': 'shippingservice-0', 'count': 60}, {'cmdb_id': 'productcatalogservice-1', 'count': 60}, {'cmdb_id': 'recommendationservice-0', 'count': 60}], 'log_name_counts_top50': [{'log_name': 'log.error_count', 'count': 884}, {'log_name': 'log.row_count', 'count': 884}]}, 'error_logs_summary': {'general': {'rows': 689, 'min_timestamp': '2022-03-20T22:30:05+08:00', 'max_timestamp': '2022-03-20T22:59:57+08:00'}, 'top_cmdb_ids': [{'cmdb_id': 'frontend-2', 'count': 463}, {'cmdb_id': 'frontend-0', 'count': 150}, {'cmdb_id': 'frontend-1', 'count': 76}], 'top_message_prefixes_100chars_top10': [{'message_prefix': 'severity: warning, message: failed to retrieve ads', 'count': 689}]}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df and tz from previous cell
# Ensure timestamp column is datetime in UTC+8
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Define candidates
candidate_cmdbs = ['frontend-2', 'frontend-0', 'frontend-1', 'adservice', 'adservice2']
node5_kpis = metric_df[metric_df['kpi_name'].str.contains('node-5', na=False)]['kpi_name'].unique().tolist()

# Build list of (component_label, kpi_name) to analyze
pairs = []

# For candidate cmdb_ids: include each unique (cmdb_id, kpi_name)
cand_df = metric_df[metric_df['cmdb_id'].isin(candidate_cmdbs)]
unique_pairs = cand_df[['cmdb_id', 'kpi_name']].drop_duplicates()
for _, r in unique_pairs.iterrows():
    pairs.append((r['cmdb_id'], r['kpi_name']))

# For node-5 kpi series: treat component label as the kpi_name itself
for k in node5_kpis:
    pairs.append((k, k))

# Deduplicate pairs while preserving order
seen = set()
pairs_dedup = []
for p in pairs:
    if p not in seen:
        seen.add(p)
        pairs_dedup.append(p)
pairs = pairs_dedup

# 2) Compute global P95, P05, full_series_count for each pair (before filtering)
records = []
for comp, kpi in pairs:
    if comp == kpi and kpi in node5_kpis:
        # node-5 series: select rows by kpi_name only
        series = metric_df.loc[metric_df['kpi_name'] == kpi, 'value'].dropna()
    else:
        # normal component-kpi pair
        series = metric_df.loc[(metric_df['cmdb_id'] == comp) & (metric_df['kpi_name'] == kpi), 'value'].dropna()
    full_count = int(series.shape[0])
    if full_count == 0:
        p95 = np.nan
        p05 = np.nan
    else:
        p95 = float(series.quantile(0.95))
        p05 = float(series.quantile(0.05))
    records.append({
        'component': comp,
        'kpi_name': kpi,
        'full_series_count': full_count,
        'p95': p95,
        'p05': p05
    })

meta_df = pd.DataFrame.from_records(records)

# 3) Filter metric_df to incident window (inclusive start, exclusive end) in UTC+8
start = pd.to_datetime('2022-03-20 22:30:00').tz_localize(tz)
end = pd.to_datetime('2022-03-20 23:00:00').tz_localize(tz)
window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] < end)].copy()

# 3b) For each pair, find anomaly points in window where value >= P95 OR value <= P05
anomaly_rows = []
for _, row in meta_df.iterrows():
    comp = row['component']
    kpi = row['kpi_name']
    p95 = row['p95']
    p05 = row['p05']
    full_count = int(row['full_series_count'])
    if pd.isna(p95) or pd.isna(p05) or full_count == 0:
        anomaly_count = 0
        earliest_ts = None
    else:
        if comp == kpi and kpi in node5_kpis:
            sel = window_df[window_df['kpi_name'] == kpi]
        else:
            sel = window_df[(window_df['cmdb_id'] == comp) & (window_df['kpi_name'] == kpi)]
        if sel.empty:
            anomaly_count = 0
            earliest_ts = None
        else:
            cond = (sel['value'] >= p95) | (sel['value'] <= p05)
            anoms = sel[cond].sort_values('ts')
            anomaly_count = int(anoms.shape[0])
            if anomaly_count > 0:
                earliest_ts = anoms['ts'].iloc[0].isoformat()
            else:
                earliest_ts = None
    severity = float(anomaly_count) / full_count if full_count > 0 else 0.0
    anomaly_rows.append({
        'component': comp,
        'kpi_name': kpi,
        'full_series_count': full_count,
        'p95': p95,
        'p05': p05,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_in_window': earliest_ts,
        'severity': severity
    })

anomaly_df = pd.DataFrame.from_records(anomaly_rows)

# 5) Sort and limit top 30 by anomaly_count desc, tie-breaker severity desc
anomaly_df_sorted = anomaly_df.sort_values(['anomaly_count_in_window', 'severity'], ascending=[False, False])
top_df = anomaly_df_sorted.head(30).reset_index(drop=True)

# Convert numeric types to native python types for JSON serialization where needed
top_df['full_series_count'] = top_df['full_series_count'].astype(int)
top_df['anomaly_count_in_window'] = top_df['anomaly_count_in_window'].astype(int)
top_df['p95'] = top_df['p95'].astype(float)
top_df['p05'] = top_df['p05'].astype(float)
top_df['severity'] = top_df['severity'].astype(float)

# Final JSON-serializable list
metric_anomalies_candidates = top_df.to_dict(orient='records')

metric_anomalies_candidates

```
Out[2]:
```
Findings from metric.csv (candidate components + node-5 KPIs):

- What was analyzed: component-KPI series for frontend-2, frontend-0, frontend-1, adservice, adservice2, plus every KPI whose name contains "node-5".

- Main result: the anomalies are concentrated in node-5 related KPIs. Many node-5 series show anomaly_count_in_window == full_series_count (severity = 1.0) with the earliest anomaly at 2022-03-20T22:30:00+08:00. Examples:
  - container.node-5.container_network_receive_errors.eth0 — full_series_count 1050, anomaly_count 1050, earliest 2022-03-20T22:30:00+08:00, severity 1.0
  - container.node-5.container_network_receive_packets_dropped.eth0 — full_series_count 1050, anomaly_count 1050, severity 1.0
  - container.node-5.container_network_transmit_errors.eth0 and _packets_dropped.eth0 — same pattern (1050 / 1050)
  - Many other node-5 KPIs (CPU throttling, CPU load average, filesystem I/O and inode metrics, memory swap, task states, etc.) — typically full_series_count 990 and anomaly_count 990 (severity 1.0). Earliest anomaly timestamp for these is also 2022-03-20T22:30:00+08:00.

- Frontend and adservice series: none of the frontend/adservice component-KPI pairs rose to the top of the anomaly list returned — the strongest signals came from node-5 KPIs.

- Interpretation & next steps:
  - Node-5 shows broad signals across network errors/packet drops and CPU/FS-related KPIs at the start of the incident window. This suggests a node-level issue (node-5) that could affect pods running there and upstream calls (consistent with frontends reporting "failed to retrieve ads" in logs/errors).
  - Recommended next steps: investigate node-5 (network interface eth0 errors and drops, CPU throttling, disk/FS conditions) and validate connectivity between frontends and adservice/adservice2; also verify the thresholding logic because many node-5 KPIs have p95/p05 = 0 (flat/mostly-zero series), which can cause every point to be flagged — confirm whether those zero-valued baselines are expected.

The original code execution output of IPython Kernel is also provided below for reference:

[{'component': 'container.node-5.container_network_receive_errors.eth0', 'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'full_series_count': 1050, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 1050, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_network_receive_packets_dropped.eth0', 'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'full_series_count': 1050, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 1050, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_network_transmit_errors.eth0', 'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'full_series_count': 1050, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 1050, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'full_series_count': 1050, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 1050, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_cpu_cfs_throttled_periods', 'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'full_series_count': 990, 'p95': 0.5, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_cpu_load_average_10s', 'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_inodes./dev/vda1', 'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_inodes_free./dev/vda1', 'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_io_current./dev/vda1', 'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_limit_MB./dev/vda1', 'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'full_series_count': 990, 'p95': 604630.73828125, 'p05': 604630.73828125, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_read_seconds./dev/vda1', 'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_reads./dev/vda1', 'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_reads_merged./dev/vda1', 'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_sector_reads./dev/vda1', 'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_sector_writes./dev/vda1', 'kpi_name': 'container.node-5.container_fs_sector_writes./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_write_seconds./dev/vda1', 'kpi_name': 'container.node-5.container_fs_write_seconds./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_writes./dev/vda1', 'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_fs_writes_merged./dev/vda1', 'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_memory_failures.container.pgmajfault', 'kpi_name': 'container.node-5.container_memory_failures.container.pgmajfault', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_memory_swap', 'kpi_name': 'container.node-5.container_memory_swap', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_spec_cpu_period', 'kpi_name': 'container.node-5.container_spec_cpu_period', 'full_series_count': 990, 'p95': 100000.0, 'p05': 100000.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_spec_memory_reservation_limit_MB', 'kpi_name': 'container.node-5.container_spec_memory_reservation_limit_MB', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_tasks_state.iowaiting', 'kpi_name': 'container.node-5.container_tasks_state.iowaiting', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_tasks_state.running', 'kpi_name': 'container.node-5.container_tasks_state.running', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_tasks_state.sleeping', 'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_tasks_state.stopped', 'kpi_name': 'container.node-5.container_tasks_state.stopped', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}, {'component': 'container.node-5.container_tasks_state.uninterruptible', 'kpi_name': 'container.node-5.container_tasks_state.uninterruptible', 'full_series_count': 990, 'p95': 0.0, 'p05': 0.0, 'anomaly_count_in_window': 990, 'earliest_anomaly_timestamp_in_window': '2022-03-20T22:30:00+08:00', 'severity': 1.0}]```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse trace_df and tz if available; otherwise load and set timezone
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure tz-aware timestamp column 'ts' in UTC+8
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive start, exclusive end)
start = pd.to_datetime('2022-03-20 22:30:00').tz_localize(tz)
end = pd.to_datetime('2022-03-20 23:00:00').tz_localize(tz)

# 1) trace_name values containing 'adservice'
ad_df = trace_df[trace_df['trace_name'].str.contains('adservice', na=False)].copy()
if not ad_df.empty:
    grp_ad = ad_df.groupby('trace_name', sort=False)
    ad_summary = grp_ad.agg(
        count=('trace_name', 'size'),
        earliest_ts_full=('ts', 'min'),
        latest_ts_full=('ts', 'max')
    ).reset_index()
    # earliest timestamp within the incident window (if any)
    def earliest_in_window(name):
        sel = ad_df[ad_df['trace_name'] == name]
        win = sel[(sel['ts'] >= start) & (sel['ts'] < end)]
        return win['ts'].min() if not win.empty else pd.NaT
    ad_summary['earliest_ts_in_window'] = ad_summary['trace_name'].apply(earliest_in_window)
    # Format timestamps to ISO strings or None
    ad_summary['earliest_ts_full'] = ad_summary['earliest_ts_full'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    ad_summary['latest_ts_full'] = ad_summary['latest_ts_full'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    ad_summary['earliest_ts_in_window'] = ad_summary['earliest_ts_in_window'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    ad_summary = ad_summary.sort_values('count', ascending=False).head(20)
    ad_summary = ad_summary[['trace_name', 'count', 'earliest_ts_full', 'latest_ts_full', 'earliest_ts_in_window']].reset_index(drop=True)
    # ensure native types
    ad_summary['count'] = ad_summary['count'].astype(int)
    traces_to_adservice = ad_summary.to_dict(orient='records')
else:
    traces_to_adservice = []

# 2) Frontend-2 related traces
f2_sel = trace_df[ trace_df['trace_name'].str.contains('frontend-2', na=False) | (trace_df['cmdb_id'] == 'frontend-2') ].copy()
if not f2_sel.empty:
    grp_f2 = f2_sel.groupby('trace_name', sort=False)
    f2_summary = grp_f2.agg(
        count=('trace_name', 'size'),
        earliest_ts_full=('ts', 'min')
    ).reset_index()
    # earliest timestamp within incident window
    def earliest_win_for(name):
        sel = f2_sel[f2_sel['trace_name'] == name]
        win = sel[(sel['ts'] >= start) & (sel['ts'] < end)]
        return win['ts'].min() if not win.empty else pd.NaT
    f2_summary['earliest_ts_in_window'] = f2_summary['trace_name'].apply(earliest_win_for)
    # error_rate stats: if trace_name contains 'error_rate' compute avg and max of value within that group
    def error_stats(name):
        if 'error_rate' in name:
            rows = f2_sel[f2_sel['trace_name'] == name]['value'].dropna()
            if rows.empty:
                return (None, None)
            return (float(rows.mean()), float(rows.max()))
        return (None, None)
    stats = f2_summary['trace_name'].apply(error_stats)
    f2_summary['error_rate_avg'] = stats.apply(lambda t: t[0])
    f2_summary['error_rate_max'] = stats.apply(lambda t: t[1])
    # Format timestamps
    f2_summary['earliest_ts_full'] = f2_summary['earliest_ts_full'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    f2_summary['earliest_ts_in_window'] = f2_summary['earliest_ts_in_window'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    f2_summary = f2_summary.sort_values('count', ascending=False).head(20)
    f2_summary = f2_summary[['trace_name', 'count', 'earliest_ts_full', 'earliest_ts_in_window', 'error_rate_avg', 'error_rate_max']].reset_index(drop=True)
    f2_summary['count'] = f2_summary['count'].astype(int)
    frontend2_trace_groups = f2_summary.to_dict(orient='records')
else:
    frontend2_trace_groups = []

result = {
    'traces_to_adservice': traces_to_adservice,
    'frontend2_trace_groups': frontend2_trace_groups
}

result

```
Out[3]:
```
Summary of trace.csv analysis (UTC+8):

1) Traces containing "adservice":
- Found active traces to adservice and adservice2 during the incident window. Key items (earliest in full series and in-window = 2022-03-20T22:30:00+08:00; latest full = 2022-03-20T22:59:00+08:00):
  - trace.to_adservice-0.{duration_mean,duration_p95,error_rate,row_count} — 90 rows each
  - trace.to_adservice-1.{duration_mean,duration_p95,error_rate,row_count} — 90 rows each
  - trace.to_adservice-2.{duration_mean,duration_p95,error_rate,row_count} — 90 rows each
  - trace.to_adservice2-0.{duration_mean,duration_p95,error_rate,row_count} — 30 rows each
- All these traces have data covering the incident window and their earliest in-window timestamps are 2022-03-20T22:30:00+08:00.

2) Frontend-2 related trace groups (top by count, limited to 20):
- The largest frontend-2 groups are trace.from_frontend-2.* with 617 rows each (duration_mean, duration_p95, error_rate, row_count). Earliest timestamps (full series and in-window) are 2022-03-20T22:30:00+08:00.
- trace.from_frontend-2.error_rate has avg = 0.0 and max = 0.0 (no elevated error_rate in that metric).
- Several trace.self.* and trace.from_root.* groups (count 30 each) also appear with earliest in-window 22:30; trace.self.error_rate and trace.from_root.error_rate show avg = 0.0 and max = 0.0.
- Frontend-2 also has trace.to_adservice-* groups (count 30 per group in the frontend-2 subset) with earliest in-window 22:30; these error_rate groups also show avg = 0.0 and max = 0.0.

Interpretation / next steps:
- Tracing shows frontend-2 actively calling adservice/adservice2 during the incident window (requests started at 22:30). Traces to adservice are present across the window.
- Error-rate metrics in the trace groups reported here are zero on average and max for the groups shown, so trace-level error rates do not show elevated numeric error fractions despite the frontend error logs ("failed to retrieve ads").
- Combine this with prior findings: frontends report failed-to-retrieve-ads warnings while node-5 shows strong node-level anomalies. Recommended next steps: investigate network/connectivity between frontend-2 and adservice instances and inspect node-5 (network interface eth0) where many node metrics were flagged.

The original code execution output of IPython Kernel is also provided below for reference:

{'traces_to_adservice': [{'trace_name': 'trace.to_adservice-0.duration_mean', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-0.duration_p95', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-0.error_rate', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-0.row_count', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-1.duration_mean', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-1.duration_p95', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-1.error_rate', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-1.row_count', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-2.duration_mean', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-2.duration_p95', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-2.error_rate', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice-2.row_count', 'count': 90, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice2-0.duration_mean', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice2-0.duration_p95', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice2-0.error_rate', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}, {'trace_name': 'trace.to_adservice2-0.row_count', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'latest_ts_full': '2022-03-20T22:59:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00'}], 'frontend2_trace_groups': [{'trace_name': 'trace.from_frontend-2.duration_mean', 'count': 617, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.from_frontend-2.duration_p95', 'count': 617, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.from_frontend-2.error_rate', 'count': 617, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': 0.0, 'error_rate_max': 0.0}, {'trace_name': 'trace.from_frontend-2.row_count', 'count': 617, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.from_root.duration_mean', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.from_root.duration_p95', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.from_root.error_rate', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': 0.0, 'error_rate_max': 0.0}, {'trace_name': 'trace.from_root.row_count', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.self.duration_mean', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.self.duration_p95', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.self.error_rate', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': 0.0, 'error_rate_max': 0.0}, {'trace_name': 'trace.self.row_count', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-0.duration_mean', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-0.duration_p95', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-0.error_rate', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': 0.0, 'error_rate_max': 0.0}, {'trace_name': 'trace.to_adservice-0.row_count', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-1.duration_mean', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-1.duration_p95', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}, {'trace_name': 'trace.to_adservice-1.error_rate', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': 0.0, 'error_rate_max': 0.0}, {'trace_name': 'trace.to_adservice-1.row_count', 'count': 30, 'earliest_ts_full': '2022-03-20T22:30:00+08:00', 'earliest_ts_in_window': '2022-03-20T22:30:00+08:00', 'error_rate_avg': nan, 'error_rate_max': nan}]}```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse metric_df and tz if available; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware datetime column 'ts' in UTC+8
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive start, exclusive end)
start = pd.to_datetime('2022-03-20 22:30:00').tz_localize(tz)
end = pd.to_datetime('2022-03-20 23:00:00').tz_localize(tz)

# Select rows where kpi_name contains 'node-5'
node5_rows = metric_df[metric_df['kpi_name'].str.contains('node-5', na=False)].copy()

# Group by cmdb_id and compute required aggregates
if not node5_rows.empty:
    grp = node5_rows.groupby('cmdb_id', sort=False)
    summary = grp.agg(
        count_of_node5_rows=('kpi_name', 'size'),
        earliest_timestamp_full=('ts', 'min')
    ).reset_index()
    # earliest timestamp within incident window (or NaT)
    def earliest_in_window(cmdb):
        sel = node5_rows[node5_rows['cmdb_id'] == cmdb]
        win = sel[(sel['ts'] >= start) & (sel['ts'] < end)]
        return win['ts'].min() if not win.empty else pd.NaT
    summary['earliest_timestamp_in_window'] = summary['cmdb_id'].apply(earliest_in_window)
    # Format timestamps to ISO strings or None
    summary['earliest_timestamp_full'] = summary['earliest_timestamp_full'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    summary['earliest_timestamp_in_window'] = summary['earliest_timestamp_in_window'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    # Keep selected columns and types
    summary = summary[['cmdb_id', 'count_of_node5_rows', 'earliest_timestamp_full', 'earliest_timestamp_in_window']]
    summary['count_of_node5_rows'] = summary['count_of_node5_rows'].astype(int)
    # Sort and limit to top 50 by count desc
    node5_cmdb_summary = summary.sort_values('count_of_node5_rows', ascending=False).head(50).reset_index(drop=True)
else:
    # empty result
    node5_cmdb_summary = pd.DataFrame(columns=['cmdb_id', 'count_of_node5_rows', 'earliest_timestamp_full', 'earliest_timestamp_in_window'])

# Presence flags for specific frontends
presence_flags = {
    'frontend-2_present': bool((node5_cmdb_summary['cmdb_id'] == 'frontend-2').any()),
    'frontend-1_present': bool((node5_cmdb_summary['cmdb_id'] == 'frontend-1').any()),
    'frontend-0_present': bool((node5_cmdb_summary['cmdb_id'] == 'frontend-0').any())
}

# Display compact results: DataFrame and presence flags
node5_cmdb_summary, presence_flags

```
Out[4]:
```
Summary of cmdb_ids that have at least one metric whose kpi_name contains "node-5":

- Total distinct cmdb_ids found: 35 (returned list limited to top 50; full result contained 35 rows).
- Most common entries (highest count_of_node5_rows = 1920) include: adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1 (and many other service pods). Many groups have identical earliest timestamps.
- Examples at the other end: paymentservice-1 has 1800 rows; redis-cart2-0 and redis-cart-0 have 240 rows.
- earliest_timestamp_in_full_series for these cmdb_ids is 2022-03-20T22:30:00+08:00 in the results shown, and for most entries there is also an earliest timestamp inside the incident window (earliest_timestamp_in_window = 2022-03-20T22:30:00+08:00). If a cmdb_id had no points in-window, the field would be null (not the case for the examples above).
- Presence of specific frontends in the list:
  - frontend-2_present: True
  - frontend-1_present: True
  - frontend-0_present: True

Interpretation: node-5-related KPIs appear across many pod/component cmdb_ids (35 distinct cmdb_ids), with most series starting at or covering the incident window start (2022-03-20T22:30:00+08:00). The three frontend pods are included among those that have node-5 KPI rows.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id  count_of_node5_rows    earliest_timestamp_full earliest_timestamp_in_window
0          adservice-0                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
1          adservice-1                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
2          adservice-2                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
3        cartservice-0                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
4        cartservice-1                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
..                 ...                  ...                        ...                          ...
30   shippingservice-0                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
31  shippingservice2-0                 1920  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
32    paymentservice-1                 1800  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
33       redis-cart2-0                  240  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00
34        redis-cart-0                  240  2022-03-20T22:30:00+08:00    2022-03-20T22:30:00+08:00

[35 rows x 4 columns], {'frontend-2_present': True, 'frontend-1_present': True, 'frontend-0_present': True})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```